# *NEXT WORD PREDICTION*

### *Problem Statement:*
The objective of this project is to build a model that predicts the next word in a sentence based on the previous words. This is a key task in Natural Language Processing, used in applications like text auto-completion and chatbots. The model uses an Long Short-Term Memory (LSTM) network to capture sequential dependencies and context in text data. The system learns from a text corpus to generate accurate and meaningful word predictions.

In [2]:
import zipfile
import numpy as np
import pandas as pd
import re
import time

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense,Bidirectional, Dropout

In [3]:
with zipfile.ZipFile(r"C:\Users\ssvmj\OneDrive\Innomatics\Data Science\Module 8 NLP\Assignments\next-word-prediction.zip") as zip_ref:
    zip_ref.extractall("data")

### *Data Loading*

In [4]:
# Load dataset
with open("data/text.txt", "r", encoding="utf-8") as f:
    text = f.read()
text = text[:150000]

print(text) 

﻿
Project Gutenberg's The Adventures of Sherlock Holmes, by Arthur Conan Doyle

This eBook is for the use of anyone anywhere at no cost and with
almost no restrictions whatsoever.  You may copy it, give it away or
re-use it under the terms of the Project Gutenberg License included
with this eBook or online at www.gutenberg.net


Title: The Adventures of Sherlock Holmes

Author: Arthur Conan Doyle

Release Date: November 29, 2002 [EBook #1661]
Last Updated: May 20, 2019

Language: English

Character set encoding: UTF-8

*** START OF THIS PROJECT GUTENBERG EBOOK THE ADVENTURES OF SHERLOCK HOLMES ***



Produced by an anonymous Project Gutenberg volunteer and Jose Menendez



cover



The Adventures of Sherlock Holmes



by Arthur Conan Doyle



Contents


   I.     A Scandal in Bohemia
   II.    The Red-Headed League
   III.   A Case of Identity
   IV.    The Boscombe Valley Mystery
   V.     The Five Orange Pips
   VI.    The Man with the Twisted Lip
   VII.   The Adventure of the Blue 

### *Data Cleaning*

Although data cleaning is typically important, for this Next Word Prediction task, minimal preprocessing preserved the natural structure and context of the language. The Bidirectional LSTM model trained on raw text performed better, as aggressive cleaning removed useful linguistic patterns. Therefore, data cleaning was intentionally skipped to retain contextual richness and improve prediction quality.

### *Data Preprocessing*

In [5]:
# Import Keras Tokenizer for converting text into numerical format
# Fit the tokenizer on the text data,This builds a vocabulary and assigns a unique index to each word
# Display the word-to-index mapping 

tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])
print(tokenizer.word_index)
print("vocab Size:",len(tokenizer.word_index))

{'the': 1, 'and': 2, 'to': 3, 'of': 4, 'a': 5, 'i': 6, '”': 7, 'in': 8, 'that': 9, 'it': 10, 'he': 11, 'was': 12, 'you': 13, 'his': 14, 'is': 15, 'as': 16, 'with': 17, 'have': 18, 'for': 19, 'at': 20, 'my': 21, 'had': 22, 'not': 23, 'which': 24, 'me': 25, 'be': 26, 'but': 27, 'holmes': 28, 'said': 29, 'from': 30, 'mr': 31, 'upon': 32, 'we': 33, 'she': 34, 'him': 35, 'your': 36, 'on': 37, 'very': 38, 'so': 39, 'an': 40, 'there': 41, 'her': 42, 'would': 43, '“i': 44, 'all': 45, 'are': 46, 'this': 47, 'when': 48, 'by': 49, 'were': 50, 'one': 51, 'what': 52, '’': 53, 'out': 54, 'then': 55, 'will': 56, 'man': 57, 'up': 58, 'do': 59, 'been': 60, 'could': 61, 'little': 62, 'or': 63, 'if': 64, 'who': 65, 'has': 66, 'no': 67, 'they': 68, 'into': 69, 'some': 70, 'down': 71, 'see': 72, 'am': 73, 'know': 74, 'more': 75, 'may': 76, 'other': 77, 'than': 78, 'did': 79, 'two': 80, 'our': 81, 'must': 82, 'over': 83, 'just': 84, 'us': 85, 'now': 86, 'come': 87, 'about': 88, 'them': 89, 'should': 90, 'on

In [6]:
# Split text into sentences,Convert sentence into sequence of word indices
# Generate n-gram sequences from each sentence
input_sequences = []

for sentence in text.split('\n'):
    tokenized_sentence = tokenizer.texts_to_sequences([sentence])[0]
    
    for i in range(1, len(tokenized_sentence)):
        n_gram_sequence = tokenized_sentence[:i+1]
        input_sequences.append(n_gram_sequence)

In [8]:
## length of the biggest line
max_len = max(len(x) for x in input_sequences)
print(max_len)

20


In [9]:
# Appling padding to all input sentences to ensure all sequences have same length
# padding='pre': adds zeros at the beginning of shorter sequences
padded_input_sequences = pad_sequences(input_sequences, maxlen=max_len, padding='pre')
display(padded_input_sequences)

array([[   0,    0,    0, ...,    0,  734, 1903],
       [   0,    0,    0, ...,  734, 1903,    1],
       [   0,    0,    0, ..., 1903,    1,  595],
       ...,
       [   0,    0,    0, ...,    9,    1,  505],
       [   0,    0,    0, ...,    1,  505,    4],
       [   0,    0,    0, ...,  505,    4, 1626]])

In [10]:
#X (input features)contains all elements of each sequence EXCEPT the last one
#[:, :-1] means:select all rows (all sequences),select all columns except the last column
X = padded_input_sequences[:,:-1]

## y(target) contains ONLY the last element of each sequence
#[:, -1] means: select all rows, select the last column.
y = padded_input_sequences[:,-1]

In [11]:
print(X)
print(X.shape)

[[   0    0    0 ...    0    0  734]
 [   0    0    0 ...    0  734 1903]
 [   0    0    0 ...  734 1903    1]
 ...
 [   0    0    0 ...  227    9    1]
 [   0    0    0 ...    9    1  505]
 [   0    0    0 ...    1  505    4]]
(26043, 19)


In [12]:
print(y)
print(y.shape)

[1903    1  595 ...  505    4 1626]
(26043,)


In [13]:
# tokenizer.word_index is a dictionary
# Define input dimension (vocabulary size for the model)
# +1 is added because index 0 is reserved for padding (no word)
# So actual input dimension = vocab size + 1 (for padding token)

print(" Total number of word: " ,len(tokenizer.word_index))
input_dim = len(tokenizer.word_index)+1

 Total number of word:  4201


### *Model Building*

- The model uses an Embedding layer to convert words into dense vector representations, allowing it to capture semantic relationships between words instead of relying on raw indices.
- Bidirectional LSTM layers are applied to learn context from both past and future words, which improves the performance of next-word prediction.
- Dropout is used to reduce overfitting and enhance the model’s generalization ability.
- Finally, a Softmax output layer with sparse categorical crossentropy is used to efficiently predict the most probable next word from the vocabulary.

In [15]:
model = Sequential()

model.add(Embedding(input_dim=input_dim, output_dim=100, input_length=max_len-1))

model.add(Bidirectional(LSTM(128, return_sequences=True)))
model.add(Dropout(0.2))

model.add(Bidirectional(LSTM(64)))

model.add(Dense(input_dim, activation='softmax'))

model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)
model.build(input_shape=(None, max_len-1))

model.summary()

C:\Users\ssvmj\anaconda3\envs\ai_env\lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding (Embedding)                │ (None, 19, 100)             │         420,200 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ bidirectional (Bidirectional)        │ (None, 19, 256)             │         234,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 19, 256)             │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ bidirectional_1 (Bidirectional)      │ (None, 128)                 │         164,352 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 4202)                │         542,058 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 1,361,106 (5.19 MB)

 Trainable params: 1,361,106 (5.19 MB)

 Non-trainable params: 0 (0.00 B)

In [16]:
model.fit(
    X, y,
    epochs=100,
    validation_split=0.2,
    verbose=1
)

Epoch 1/100
652/652 ━━━━━━━━━━━━━━━━━━━━ 29s 33ms/step - accuracy: 0.0493 - loss: 6.4652 - val_accuracy: 0.0561 - val_loss: 6.3936
Epoch 2/100
652/652 ━━━━━━━━━━━━━━━━━━━━ 21s 33ms/step - accuracy: 0.0583 - loss: 5.9891 - val_accuracy: 0.0664 - val_loss: 6.4332
Epoch 3/100
652/652 ━━━━━━━━━━━━━━━━━━━━ 21s 33ms/step - accuracy: 0.0676 - loss: 5.8060 - val_accuracy: 0.0697 - val_loss: 6.4244
Epoch 4/100
652/652 ━━━━━━━━━━━━━━━━━━━━ 21s 32ms/step - accuracy: 0.0845 - loss: 5.6256 - val_accuracy: 0.0825 - val_loss: 6.4736
Epoch 5/100
652/652 ━━━━━━━━━━━━━━━━━━━━ 21s 33ms/step - accuracy: 0.1030 - loss: 5.4514 - val_accuracy: 0.0850 - val_loss: 6.4589
Epoch 6/100
652/652 ━━━━━━━━━━━━━━━━━━━━ 21s 33ms/step - accuracy: 0.1146 - loss: 5.2965 - val_accuracy: 0.0893 - val_loss: 6.4594
Epoch 7/100
652/652 ━━━━━━━━━━━━━━━━━━━━ 21s 33ms/step - accuracy: 0.1234 - loss: 5.1586 - val_accuracy: 0.0887 - val_loss: 6.5367
Epoch 8/100
652/652 ━━━━━━━━━━━━━━━━━━━━ 22s 33ms/step - accuracy: 0.1324 - loss: 5

### *Model Saving*

In [17]:
model.save("model.h5")       

In [18]:
model.save('nextwordmodel.keras')

In [19]:
import pickle

with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

In [20]:
with open("max_len.pkl", "wb") as f:
    pickle.dump(max_len, f)

### *Model Texting*

In [21]:
import time
text = "To Sherlock Holmes"

for i in range(10):
  # tokenize
  token_text = tokenizer.texts_to_sequences([text])[0]
  # padding
  padded_token_text = pad_sequences([token_text], maxlen=max_len, padding='pre')
  # predict
  pos = np.argmax(model.predict(padded_token_text))

  for word,index in tokenizer.word_index.items():
    if index == pos:
      text = text + " " + word
      print(text)
      time.sleep(2)

1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
To Sherlock Holmes she
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step
To Sherlock Holmes she is
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 98ms/step
To Sherlock Holmes she is always
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step
To Sherlock Holmes she is always the
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 92ms/step
To Sherlock Holmes she is always the woman
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step
To Sherlock Holmes she is always the woman i
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 107ms/step
To Sherlock Holmes she is always the woman i have
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 96ms/step
To Sherlock Holmes she is always the woman i have seldom
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 101ms/step
To Sherlock Holmes she is always the woman i have seldom heard
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 99ms/step
To Sherlock Holmes she is always the woman i have seldom heard him


#### *Original Text*:
To Sherlock Holmes she is always _the_ woman. I have seldom heard him
mention her under any other name.

In [22]:
text = "“Project Gutenberg's The Adventures of”"

for i in range(10):
  # tokenize
  token_text = tokenizer.texts_to_sequences([text])[0]
  # padding
  padded_token_text = pad_sequences([token_text], maxlen=max_len, padding='pre')
  # predict
  pos = np.argmax(model.predict(padded_token_text))

  for word,index in tokenizer.word_index.items():
    if index == pos:
      text = text + " " + word
      print(text)
      time.sleep(2)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step
“Project Gutenberg's The Adventures of” of
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step
“Project Gutenberg's The Adventures of” of sherlock
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 92ms/step
“Project Gutenberg's The Adventures of” of sherlock holmes
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 100ms/step
“Project Gutenberg's The Adventures of” of sherlock holmes by
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step
“Project Gutenberg's The Adventures of” of sherlock holmes by arthur
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step
“Project Gutenberg's The Adventures of” of sherlock holmes by arthur conan
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step
“Project Gutenberg's The Adventures of” of sherlock holmes by arthur conan doyle
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step
“Project Gutenberg's The Adventures of” of sherlock holmes by arthur conan doyle at
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 98ms/step
“Project Gutenberg's The Adventures of” of sherlock holmes by arthur conan doyle at each
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step
“Proj

#### *Original Text:*
Project Gutenberg's The Adventures of Sherlock Holmes, by Arthur Conan Doyle
This eBook is for the use of anyone anywhere at no cost and with
almost no restrictions whatsoever.

In [23]:
text = "“I was still balancing"

for i in range(10):
  # tokenize
  token_text = tokenizer.texts_to_sequences([text])[0]
  # padding
  padded_token_text = pad_sequences([token_text], maxlen=max_len, padding='pre')
  # predict
  pos = np.argmax(model.predict(padded_token_text))

  for word,index in tokenizer.word_index.items():
    if index == pos:
      text = text + " " + word
      print(text)
      time.sleep(2)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 108ms/step
“I was still balancing the
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 92ms/step
“I was still balancing the matter
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 112ms/step
“I was still balancing the matter in
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 97ms/step
“I was still balancing the matter in my
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 91ms/step
“I was still balancing the matter in my mind
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 143ms/step
“I was still balancing the matter in my mind when
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 95ms/step
“I was still balancing the matter in my mind when a
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step
“I was still balancing the matter in my mind when a hansom
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step
“I was still balancing the matter in my mind when a hansom cab
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step
“I was still balancing the matter in my mind when a hansom cab drove


#### *Original Text:*
“I was still balancing the matter in my mind when a hansom cab drove up
to Briony Lodge, and a gentleman sprang out.

In [24]:
text = "“Pray take a seat,"

for i in range(10):
  # tokenize
  token_text = tokenizer.texts_to_sequences([text])[0]
  # padding
  padded_token_text = pad_sequences([token_text], maxlen=max_len, padding='pre')
  # predict
  pos = np.argmax(model.predict(padded_token_text))

  for word,index in tokenizer.word_index.items():
    if index == pos:
      text = text + " " + word
      print(text)
      time.sleep(2)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step
“Pray take a seat, ”
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step
“Pray take a seat, ” said
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step
“Pray take a seat, ” said holmes
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step
“Pray take a seat, ” said holmes “this
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step
“Pray take a seat, ” said holmes “this is
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 88ms/step
“Pray take a seat, ” said holmes “this is my
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step
“Pray take a seat, ” said holmes “this is my friend
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step
“Pray take a seat, ” said holmes “this is my friend and
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step
“Pray take a seat, ” said holmes “this is my friend and colleague
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step
“Pray take a seat, ” said holmes “this is my friend and colleague dr


#### *Original Text:*
“Pray take a seat,” said Holmes. “This is my friend and colleague, Dr.
Watson, who is occasionally good enough to help me in my cases. Whom
have I the honour to address?

### *Conclusion*
This project builds a next-word prediction model using NLP and deep learning techniques. Text data was preprocessed and sequences were generated to train a Bidirectional LSTM model for better context understanding. The model was able to predict the next word with reasonable accuracy. This demonstrates how deep learning can be effectively used for text prediction tasks and can be improved further with larger datasets.